In [25]:
import os, requests, json, pyodbc, logging, time
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
from sqlalchemy import create_engine
from tqdm import tqdm
from datetime import datetime, timedelta
import unicodedata
import numpy as np


load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

API_KEY = os.getenv("INAGENT_API_KEY")
ENDPOINT = os.getenv("INAGENT_URL")

CREW_MAPPING = {
    os.getenv("INAGENT_CREW_ID"): "DENTAL",
    os.getenv("INAGENT_CREW_ID2"): "OMV"
}
CREW_IDS = list(CREW_MAPPING.keys())


In [26]:
hoy = datetime.now()
hace_una_hora = hoy - timedelta(minutes=60)

env_start = os.getenv("START_DATE")
env_end = os.getenv("END_DATE")

if env_start:
    start_date = env_start
else:
    start_date = hace_una_hora.strftime("%Y-%m-%dT%H:%M:%S")

if env_end:
    end_date = env_end
else:
    end_date = hoy.strftime("%Y-%m-%dT%H:%M:%S")

print(f"Extrayendo datos desde {start_date} hasta {end_date}")


Extrayendo datos desde 2026-03-15T00:00:00 hasta 2026-05-05T14:31:17


In [27]:
def to_unix_ms(iso_date):
    dt = datetime.fromisoformat(iso_date).replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)

def safe_json_parse(val):
    try:
        return json.loads(val) if (val and val != 'null') else {}
    except:
        return {}

print(" Herramientas listas: to_unix_ms y safe_json_parse.")

 Herramientas listas: to_unix_ms y safe_json_parse.


In [28]:
all_data = []
page_size = 100 # Límite recomendado por la API [cite: 141]

for crew in CREW_IDS:
    page = 0
    label_origen = CREW_MAPPING.get(crew, "DESCONOCIDO")
    logger.info(f"Extrayendo datos de {label_origen} (ID: {crew})")
    
    while True:
        params = {
            "crew_id": crew,
            "start_ts": to_unix_ms(start_date),
            "end_ts": to_unix_ms(end_date),
            "page": page,
            "pageSize": page_size
        }
        
        headers = {"apikey": API_KEY}
        res = requests.get(ENDPOINT, headers=headers, params=params)
        
        if res.status_code != 200:
            logger.error(f"Fallo en {label_origen}, Página {page}: {res.text}")
            break
            
        data_payload = res.json().get("data", {})
        rows = data_payload.get("rows", [])
        
        if page == 0 and crew == CREW_IDS[0]:
            cols = data_payload.get("dataSchema", {}).get("columnNames", [])
            if "Origen" not in cols:
                cols.append("Origen")
        
        if not rows:
            break
            
        for row in rows:
            row.append(label_origen)
            
        all_data.extend(rows)
        logger.info(f"Página {page} de {label_origen} lista. Total: {len(all_data)}")
        
        if len(rows) < page_size: # Fin de datos para este equipo [cite: 262]
            break
        page += 1

df_raw = pd.DataFrame(all_data, columns=cols)


2026-05-05 14:31:17,453 - INFO - Extrayendo datos de DENTAL (ID: 766cddc2-eaf9-464e-8f6d-8854ef927ff3)
2026-05-05 14:31:19,572 - INFO - Página 0 de DENTAL lista. Total: 100
2026-05-05 14:31:21,183 - INFO - Página 1 de DENTAL lista. Total: 200
2026-05-05 14:31:22,527 - INFO - Página 2 de DENTAL lista. Total: 300
2026-05-05 14:31:23,642 - INFO - Página 3 de DENTAL lista. Total: 362
2026-05-05 14:31:23,643 - INFO - Extrayendo datos de OMV (ID: 2539ab63-c408-446a-b13e-1eef4f7c1ba3)
2026-05-05 14:31:25,089 - INFO - Página 0 de OMV lista. Total: 462
2026-05-05 14:31:25,792 - INFO - Página 1 de OMV lista. Total: 467


In [29]:
native_whitelist = [
    'Id', 
    'Id Externo',
    'Id Canal',
    'Canal',
    'Timestamp',
    'Inicio',
    'Fin',
    'Duración (s)', 
    'Análisis Sentimental',
    'Tema general de la conversación',
    #'Resumen',
    'Fue resuelta',
    'Fue solo agradecimiento',
    'Herramientas Usadas',
    'Es saliente',
    'Fue abandonada',
    'Contexto',
    'Origen'
]

In [30]:
context_whitelist = [
    'toolLogs',
    'tarjeta',
    'proxyData_sip_attributes_sip_trunkPhoneNumber',
    'proxyData_sip_attributes_sip_h_x-tarjeta-id',
    'comb_resume'
]

In [31]:
df_native = df_raw[[c for c in native_whitelist if c in df_raw.columns]].copy()

ctx_raw = pd.json_normalize(df_raw['Contexto'].apply(safe_json_parse))
ctx_raw.columns = [c.replace(".", "_") for c in ctx_raw.columns]

In [32]:
ctx_raw.to_csv("context.csv",index=False)

In [33]:
ctx_selected = ctx_raw[[c for c in context_whitelist if c in ctx_raw.columns]].add_prefix('ctx_')
df_base = pd.concat([df_native, ctx_selected], axis=1)
print(f"{df_base.shape}")

(467, 22)


In [34]:
col_logs = 'ctx_toolLogs'

df_tools_exploded = df_base[['Id', col_logs]].dropna(subset=[col_logs]).explode(col_logs)
tool_rows = df_tools_exploded[col_logs].apply(lambda x: x if isinstance(x, (dict, list)) else safe_json_parse(x)).tolist()
df_tools_flat = pd.json_normalize(tool_rows)

In [35]:
df_tools_flat.to_csv("tools.csv",index=False)

In [36]:
tool_whitelist = [
    'URL_fetch', 'body_fetch', 'return_fetch', 'tool', 'status', 
    'timestamp', 'code_fetch','return_fetch.msg',
    'return_fetch.message','return_fetch.data.client_complete_name',
    'return_fetch.data.policy_number','return_fetch.data.program_name',
    'return_fetch.data.program_status',
    'return_fetch.data.client_account',
    'return_fetch.data.client_telefono',
    'return_fetch.success',
    'return_fetch.data.client_card',
    'body_fetch.cancelMotive',
    'body_fetch.scheduleDate', #date
    'body_fetch.specialty',
    'return_fetch.response.nameDoctor',
    'return_fetch.response.name_service',
    'return_fetch.response.creationDate' #Timestrap
    'return_fetch.response.title',
    'return_fetch.response.provider',
    'return_fetch.response.Kinship',
    'return_fetch.response.category',
    'return_fetch.response.nameDoctor',
    'body_fetch.cas',
    'return_fetch.response.cas_folio', 'return_fetch.cas_folio',
]


In [37]:
cols_t = [c for c in tool_whitelist if c in df_tools_flat.columns]
df_tools_final = df_tools_flat[cols_t].copy()
df_tools_final.columns = [f"tool_{c.replace('.', '_')}" for c in df_tools_final.columns]

df_tools_final.index = df_tools_exploded.index
df_tools_merged = pd.concat([df_tools_exploded[['Id']], df_tools_final], axis=1)

print(f'{df_tools_merged.shape}')

(2196, 30)


In [38]:
df_tools_final.info()

<class 'pandas.DataFrame'>
Index: 2196 entries, 0 to 464
Data columns (total 29 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   tool_URL_fetch                               2196 non-null   str    
 1   tool_body_fetch                              182 non-null    str    
 2   tool_return_fetch                            514 non-null    str    
 3   tool_tool                                    2196 non-null   str    
 4   tool_status                                  2196 non-null   str    
 5   tool_timestamp                               2196 non-null   str    
 6   tool_code_fetch                              1667 non-null   float64
 7   tool_return_fetch_msg                        392 non-null    str    
 8   tool_return_fetch_message                    1008 non-null   str    
 9   tool_return_fetch_data_client_complete_name  392 non-null    str    
 10  tool_return_fetch

In [39]:
df_final = df_base.merge(df_tools_merged, on='Id', how='left')
df_final.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_") for c in df_final.columns]

In [40]:
def aplicar_ancla_maestra(df): 
    df['tool_timestamp_dt'] = pd.to_datetime(df['tool_timestamp'], errors='coerce')
    df = df.sort_values(by=['Id', 'tool_timestamp_dt'], ascending=[True, False])
    
    es_el_ancla = ~df.duplicated(subset=['Id'], keep='first')
    
    df['interaccion_unica'] = np.where(es_el_ancla,1,0)
    df['Estatus_Final'] = np.where(es_el_ancla, df['tool_tool'], None)
    
    return df

df_preparado = aplicar_ancla_maestra(df_final)

In [41]:
df_preparado.info()

<class 'pandas.DataFrame'>
Index: 2243 entries, 1475 to 1351
Data columns (total 54 columns):
 #   Column                                             Non-Null Count  Dtype              
---  ------                                             --------------  -----              
 0   Id                                                 2243 non-null   str                
 1   Id_Externo                                         2243 non-null   str                
 2   Id_Canal                                           2127 non-null   str                
 3   Canal                                              2243 non-null   str                
 4   Timestamp                                          2243 non-null   str                
 5   Inicio                                             2243 non-null   str                
 6   Fin                                                2243 non-null   str                
 7   Duración_s                                         2243 non-null   int64 

In [42]:
def crear_id_compuesto_pro(df):
    logger.info(" Generando identificadores únicos...")
    

    df['tool_tool'] = df['tool_tool'].fillna('SIN_HERRAMIENTA') # <--- CLAVE
    
    tool = df['tool_timestamp'].astype(str)
    
    df['id_registro'] = (
        df['Id'].astype(str) + "_" + 
        df['tool_tool'].astype(str) + "_" + 
        tool
    )
    return df

In [43]:
crear_id_compuesto_pro(df_preparado)

2026-05-05 14:31:26,025 - INFO -  Generando identificadores únicos...


,Id,Id_Externo,Id_Canal,Canal,Timestamp,Inicio,Fin,Duración_s,Análisis_Sentimental,Tema_general_de_la_conversación,...,tool_return_fetch_response_Kinship,tool_return_fetch_response_category,tool_return_fetch_response_nameDoctor,tool_body_fetch_cas,tool_return_fetch_response_cas_folio,tool_return_fetch_cas_folio,tool_timestamp_dt,interaccion_unica,Estatus_Final,id_registro
1475,00383c15-e33b-4c2d-9754-5d45ed727d2d,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777658487000,2026-05-01T18:01:27.000Z,2026-05-01T18:05:13.294Z,226,neutral,PIF Transferencia,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-01 18:02:04.665000+00:00,1,TransferenciaAsesor,00383c15-e33b-4c2d-9754-5d45ed727d2d_Transfere...
1474,00383c15-e33b-4c2d-9754-5d45ed727d2d,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777658487000,2026-05-01T18:01:27.000Z,2026-05-01T18:05:13.294Z,226,neutral,PIF Transferencia,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-01 18:01:41.899000+00:00,0,NaN,00383c15-e33b-4c2d-9754-5d45ed727d2d_inicializ...
1821,00d0ed11-7982-4276-bea7-0b38e58d9a5a,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777412725000,2026-04-28T21:45:25.000Z,2026-04-28T21:47:23.783Z,118,neutral,cancelar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-28 21:47:14.568000+00:00,1,CerrarSesionPorTimeout,00d0ed11-7982-4276-bea7-0b38e58d9a5a_CerrarSes...
1820,00d0ed11-7982-4276-bea7-0b38e58d9a5a,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777412725000,2026-04-28T21:45:25.000Z,2026-04-28T21:47:23.783Z,118,neutral,cancelar cita,...,NaN,NaN,NaN,cas347196,NaN,NaN,2026-04-28 21:46:29.216000+00:00,0,NaN,00d0ed11-7982-4276-bea7-0b38e58d9a5a_cancelar_...
1819,00d0ed11-7982-4276-bea7-0b38e58d9a5a,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777412725000,2026-04-28T21:45:25.000Z,2026-04-28T21:47:23.783Z,118,neutral,cancelar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-28 21:46:13.623000+00:00,0,NaN,00d0ed11-7982-4276-bea7-0b38e58d9a5a_get_fecha...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1355,fff114d6-eccf-4c08-9c2b-198623c0912c,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777668171000,2026-05-01T20:42:51.000Z,2026-05-01T21:06:27.148Z,1416,neutral,Agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-01 20:51:52.738000+00:00,0,NaN,fff114d6-eccf-4c08-9c2b-198623c0912c_enviar_a_...
1354,fff114d6-eccf-4c08-9c2b-198623c0912c,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777668171000,2026-05-01T20:42:51.000Z,2026-05-01T21:06:27.148Z,1416,neutral,Agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-01 20:50:55.939000+00:00,0,NaN,fff114d6-eccf-4c08-9c2b-198623c0912c_get_cel_n...
1353,fff114d6-eccf-4c08-9c2b-198623c0912c,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777668171000,2026-05-01T20:42:51.000Z,2026-05-01T21:06:27.148Z,1416,neutral,Agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-01 20:44:46.077000+00:00,0,NaN,fff114d6-eccf-4c08-9c2b-198623c0912c_get_benef...
1352,fff114d6-eccf-4c08-9c2b-198623c0912c,,d6aef35f-bbaf-44c9-b960-b31cff41b875,Livekit_SipCall,1777668171000,2026-05-01T20:42:51.000Z,2026-05-01T21:06:27.148Z,1416,neutral,Agendar cita,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-01 20:44:04.168000+00:00,0,NaN,fff114d6-eccf-4c08-9c2b-198623c0912c_get_servi...


In [44]:
df_preparado.to_csv("prepare.csv",index=False)

In [45]:
columnas_sql_reales = [
    'Id', 
    'Id_Externo', 
    'Id_Canal', 
    'Canal', 
    'Timestamp', 
    'Inicio', 
    'Fin', 
    'Duración_s', 
    'Análisis_Sentimental', 
    'Tema_general_de_la_conversación', 
    'Fue_resuelta', 
    'Fue_solo_agradecimiento', 
    'Herramientas_Usadas', 
    'Es_saliente', 
    'Fue_abandonada', 
    'Origen', 
    'ctx_tarjeta', 
    'ctx_proxyData_sip_attributes_sip_trunkPhoneNumber', 
    'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id',
    'ctx_comb_resume', 
    'tool_URL_fetch', 
    'tool_body_fetch', 
    'tool_return_fetch', 
    'tool_tool', 
    'tool_status', 
    'tool_timestamp', 
    'tool_code_fetch', 
    'tool_return_fetch_msg', 
    'tool_return_fetch_message', 
    'tool_return_fetch_data_client_complete_name', 
    'tool_return_fetch_data_policy_number', 
    'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_program_status', 
    'tool_return_fetch_data_client_account', 
    'tool_return_fetch_data_client_telefono', 
    'tool_return_fetch_success', 
    'tool_return_fetch_data_client_card', 
    'tool_body_fetch_scheduleDate', 
    'tool_body_fetch_specialty', 
    'tool_return_fetch_response_nameDoctor', 
    'tool_return_fetch_response_name_service', 
    'tool_return_fetch_response_provider', 
    'tool_return_fetch_response_Kinship', 
    'tool_return_fetch_response_category', 
    'tool_timestamp_dt', #Fecha
    'interaccion_unica', #Bin 1 y 0
    'Estatus_Final', #STR Varchar 
    'tool_body_fetch_cas', 
    'tool_body_fetch_cancelMotive',
    'tool_return_fetch_cas_folio', 
    'tool_return_fetch_response_cas_folio', 
    'id_registro'
]

In [46]:
def cas(df):
    columnas_cas = ['tool_return_fetch_cas_folio', 'tool_body_fetch_cas', 'tool_return_fetch_response_cas_folio']
    for col in columnas_cas:
        if col not in df.columns:
            df[col] = np.nan
        else:
            # LIMPIEZA CLAVE: Convierte strings basura en NaNs reales de Pandas
            df[col] = df[col].astype(str).replace(['n.n','nan', 'None', 'NaN', 'null', ''], np.nan)

    condiciones = [
        df['tool_return_fetch_cas_folio'].astype(str).str.contains('cas', case=False, na=False),
        df['tool_body_fetch_cas'].notna(),
        df['tool_return_fetch_response_cas_folio'].notna(),
        df['tool_return_fetch_cas_folio'].notna()
    ]

    valores = [
        df['tool_return_fetch_cas_folio'],
        df['tool_body_fetch_cas'],
        df['tool_return_fetch_response_cas_folio'],
        df['tool_return_fetch_cas_folio']
    ]

    df['cas'] = np.select(condiciones, valores, default='SIN FOLIO CAS')
    
    # Limpieza final para asegurar que no se escape ningún 'NAN' como texto
    df['cas'] = df['cas'].astype(str).str.upper().replace(['NAN', 'NONE', 'NULL'], 'SIN FOLIO CAS')

    return df

In [47]:
def pipeline_maestro_final(df, whitelist):
    df_sql = df.copy()
    
    def limpiar_nombres(txt):
        if not isinstance(txt, str): return txt
        txt = "".join(c for c in unicodedata.normalize('NFD', txt) if unicodedata.category(c) != 'Mn')
        return txt.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_")

    df_sql.columns = [limpiar_nombres(c) for c in df_sql.columns]
    whitelist_limpia = [limpiar_nombres(c) for c in whitelist]

    if 'Origen' in df_sql.columns:
        df_sql['Id_Canal'] = df_sql['Origen']
        
        if 'id_registro' in df_sql.columns:
            df_sql['Id_Externo'] = df_sql['id_registro'] 
            
        if 'tool_return_fetch_cas_folio' in df_sql.columns:
            df_sql['tool_return_fetch_response_nameDoctor'] = df_sql['tool_return_fetch_cas_folio']
        
        if 'interaccion_unica' in df_sql.columns:
            df_sql['tool_return_fetch_response_costPIFDoctor'] = df_sql['interaccion_unica']
        
        if 'Estatus_Final' in df_sql.columns:
            df_sql['tool_return_fetch_response_typeOfService'] = df_sql['Estatus_Final']


    df_sql = cas(df_sql)
    
    if 'cas' not in whitelist_limpia:
        whitelist_limpia.append('cas')

    df_sql = df_sql[[c for c in whitelist_limpia if c in df_sql.columns]]

    cols_num = [
        'Duracion_s', 'Herramientas_Usadas', 'tool_code_fetch',
        'tool_return_fetch_httpCode', 'tool_return_fetch_response_costPIFDoctor',
        'tool_return_fetch_response_selected_dentist'
    ]
    
    cols_date = ['Inicio', 'Fin', 'tool_timestamp_dt']

    for col in df_sql.columns:
        if col in cols_date:
            df_sql[col] = pd.to_datetime(df_sql[col], errors='coerce')
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif col in cols_num:
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif any(x in col for x in ['Fue_', 'Es_', 'Cerrada_']):
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce').fillna(0).astype(int)
            
        else:
            df_sql[col] = df_sql[col].astype(str).replace(['n.n','nan', 'None', 'NaN', 'null'], None)
            df_sql[col] = df_sql[col].where(df_sql[col].notnull(), None)
            
            if col in ['tool_return_fetch_response_nameDoctor', 'tool_return_fetch_response_name_service', 'tool_return_fetch_data_client_complete_name', 'ctx_proxyData_roomName']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:255] if isinstance(x, str) else x)
            elif col in ['Estatus_Final', 'Id_Externo', 'Id', 'Id_Canal', 'ctx_tarjeta', 'tool_tool']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:100] if isinstance(x, str) else x)
            elif col in ['Origen', 'Canal', 'tool_status', 'tool_return_fetch_status']:
                df_sql[col] = df_sql[col].apply(lambda x: x[:50] if isinstance(x, str) else x)

    return df_sql

In [48]:
df_listo = pipeline_maestro_final(df_preparado, columnas_sql_reales)

In [49]:
df_listo.rename(columns={
        'ctx_proxyData_sip_attributes_sip_h_x-tarjeta-id': 'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id'
    }, inplace=True)

In [50]:
para_SQL = [
    'Id', 
    'Id_Externo', 
    'Id_Canal', 
    'Canal',
    'cas', 
    'Timestamp', 
    'Inicio', 
    'Fin', 
    'Duracion_s', 
    'Analisis_Sentimental', 
    'Tema_general_de_la_conversacion', 
    'Fue_resuelta', 
    'Fue_solo_agradecimiento', 
    'Herramientas_Usadas', 
    'Es_saliente', 
    'Fue_abandonada', 
    'Origen', 
    'ctx_tarjeta', 
    'ctx_proxyData_sip_attributes_sip_trunkPhoneNumber', 
    'ctx_proxyData_sip_attributes_sip_h_x_tarjeta_id', 
    #'ctx_comb_resume',
    'tool_URL_fetch', 
    'tool_body_fetch', 
    'tool_return_fetch', 
    'tool_tool', 
    'tool_status', 
    'tool_timestamp', 
    'tool_code_fetch', 
    'tool_return_fetch_msg', 
    'tool_return_fetch_message', 
    'tool_return_fetch_data_client_complete_name', 
    'tool_return_fetch_data_policy_number', 
    'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_program_status', 
    'tool_return_fetch_data_client_account', 
    'tool_return_fetch_data_client_telefono', 
    'tool_return_fetch_success', 
    'tool_return_fetch_data_client_card', 
    'tool_body_fetch_cas', 
    'tool_body_fetch_cancelMotive', 
    'tool_return_fetch_response_cas_folio', 
    'tool_return_fetch_cas_folio',
    'tool_body_fetch_scheduleDate', 
    'tool_body_fetch_specialty', 
    'tool_return_fetch_response_nameDoctor', 
    'tool_return_fetch_response_name_service', 
    'tool_return_fetch_response_provider', 
    'tool_return_fetch_response_Kinship', 
    'tool_return_fetch_response_category', 
    'tool_timestamp_dt', 
    'interaccion_unica', 
    'Estatus_Final',
]

In [51]:
df_produccion = df_listo[[c for c in para_SQL if c in df_listo.columns]].copy()

df_produccion = df_produccion.loc[:, ~df_produccion.columns.duplicated()].copy()

duplicadas = df_produccion.columns[df_produccion.columns.duplicated()].tolist()
logger.info(f"Columnas duplicadas aniquiladas: {duplicadas}")

2026-05-05 14:31:26,461 - INFO - Columnas duplicadas aniquiladas: []


In [52]:
df_para_sql = df_produccion.copy()

TABLE_NAME = "dbo.inagent" #

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')},{os.getenv('DB_PORT')};"
    f"DATABASE={os.getenv('BD')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASS')}"
)

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    cursor.fast_executemany = True 

    cursor.execute(f"IF OBJECT_ID('tempdb..#stg_inagent') IS NOT NULL DROP TABLE #stg_inagent")
    
    cols = df_para_sql.columns.tolist()
    col_names_bracketed = ", ".join(f"[{c}]" for c in cols)
    
    cursor.execute(f"IF OBJECT_ID('tempdb..#stg_inagent') IS NOT NULL DROP TABLE #stg_inagent")
    cursor.execute(f"SELECT TOP 0 {col_names_bracketed} INTO #stg_inagent FROM {TABLE_NAME}") 

    placeholders = ", ".join("?" for _ in cols)
    sql_insert = f"INSERT INTO #stg_inagent ({col_names_bracketed}) VALUES ({placeholders})"
    
    # --- LIMPIEZA DE EMERGENCIA ANTES DE CARGAR ---
    # 1. Truncar columnas problemáticas (Error 510 buffer)
    cols_to_trim = ['tool_return_fetch_response_nameDoctor', 'tool_return_fetch_response_name_service', 
                    'tool_return_fetch_data_client_complete_name', 'tool_body_fetch', 'tool_return_fetch', 'ctx_comb_resume']
    for c in cols_to_trim:
        if c in df_para_sql.columns:
            df_para_sql[c] = df_para_sql[c].astype(str).str.slice(0, 255)
            df_para_sql[c] = df_para_sql[c].replace(['None', 'nan', 'NaN', 'null'], None)

    # 2. Eliminar id_registro si existe (Error uniqueidentifier)
    if 'id_registro' in df_para_sql.columns:
        df_para_sql = df_para_sql.drop(columns=['id_registro'])

    # 3. Deduplicar (Error MERGE duplicate)
    df_para_sql = df_para_sql.drop_duplicates(subset=['Id_Externo']).copy()

    # 4. Actualizar lista de columnas por si eliminamos alguna
    cols = df_para_sql.columns.tolist()
    col_names_bracketed = ", ".join(f"[{c}]" for c in cols)
    placeholders = ", ".join("?" for _ in cols)
    sql_insert = f"INSERT INTO #stg_inagent ({col_names_bracketed}) VALUES ({placeholders})"
    
    data_to_load = [tuple(x) for x in df_para_sql.values]
    
    logger.info(f" Subiendo {len(data_to_load)} registros a Staging...")
    cursor.executemany(sql_insert, data_to_load)
    
    sql_merge = f"""
    MERGE {TABLE_NAME} AS target
    USING #stg_inagent AS source
    ON (target.Id_Externo= source.Id_Externo)
    WHEN MATCHED THEN
        UPDATE SET 
            target.Analisis_Sentimental = source.Analisis_Sentimental,
            target.Tema_general_de_la_conversacion = source.Tema_general_de_la_conversacion,
            target.tool_status = source.tool_status,
            target.tool_return_fetch_message = source.tool_return_fetch_message,
            target.cas = source.cas
    WHEN NOT MATCHED THEN
        INSERT ({col_names_bracketed})
        VALUES ({', '.join(f'source.[{c}]' for c in cols)});
    """
    
    logger.info("Ejecutando MERGE en tabla definitiva...")
    cursor.execute(sql_merge)
    conn.commit()
    logger.info(f"ÉXITO: {len(df_para_sql)} registros sincronizados correctamente.")

except Exception as e:
    if 'conn' in locals(): conn.rollback()
    logger.error(f"Error en SQL: {e}")
finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals(): conn.close()


2026-05-05 14:31:27,498 - INFO -  Subiendo 2126 registros a Staging...
2026-05-05 14:33:33,745 - INFO - Ejecutando MERGE en tabla definitiva...
2026-05-05 14:33:34,251 - INFO - ÉXITO: 2126 registros sincronizados correctamente.


In [53]:
tools_master_whitelist = [
    'inicializar_sesion',           # Obtiene información del cliente [cite: 47]
    'CerrarConversacionPorUsuario', # Cierre por decisión del usuario [cite: 48]
    'CerrarSesionPorTimeout',       # Cierre por inactividad [cite: 49]
    'get_fecha',                    # Obtiene fecha actual [cite: 50]
    'upsert_beneficiario',          # Registra/actualiza beneficiario [cite: 51]
    'get_beneficiarios',            # Obtiene lista de beneficiarios [cite: 52]
    'get_servicios_siniestralidad', # Consulta de servicios [cite: 53]
    'validar_tarjeta',              # Consulta beneficios [cite: 54]
    'TransferenciaAsesor',          # Envío a agente humano [cite: 113] 
    'get_titular',                  # Obtiene ID del titular [cite: 55]
    'get_horarios',                 # Consulta disponibilidad [cite: 56]
    'cancelar_cita',                # Cancela cita agendada [cite: 57]
    'set_checkup',                  # Crea nueva cita [cite: 58]
    'set_context'                   # Crea contexto con Talkdesk [cite: 59]
]